# CelebA-HQ — Stage 5 metadata-only loader/split binding

Run with **Accelerator = None**. Attach the DIRE input Dataset, the code Dataset version containing `dire_modal/`, and the Stage 3 v2 split artifact (or keep its JSON under `/kaggle/working`). This cell validates paths/counts/fingerprint only: it does not open image contents, create tensors, use GPU, or train. The complete metadata scan can take time; the latest loader package prints progress during path indexing and fingerprinting. Older attached package versions still work, but may remain quiet until the scan completes.

The output report is written only to `/kaggle/working/celebahq-data-binding-report.json`; the mounted input remains read-only. Do not index/iterate the returned datasets in this stage.

In [ ]:
from pathlib import Path
import sys

INPUT_ROOT = Path('/kaggle/input')
DIRE_ROOT = Path('/kaggle/input/datasets/j4cknguyn/diffusionforensics/dire')
SPLIT_ARTIFACT = Path('/kaggle/working/celebahq-validation-split-v2-81dcda99aeb9.json')

# Kaggle mounts attached Datasets but does not add their Python packages to sys.path.
# Search only mount roots (and optional src/), never recurse through the image tree.
mount_roots = []
for mount in INPUT_ROOT.iterdir():
    if mount.name == 'datasets' and mount.is_dir():
        for owner in mount.iterdir():
            if owner.is_dir():
                mount_roots.extend(dataset for dataset in owner.iterdir() if dataset.is_dir())
    elif mount.is_dir():
        mount_roots.append(mount)
code_roots = [
    root for mount in mount_roots for root in (mount, mount / 'src')
    if (root / 'dire_modal' / '__init__.py').is_file()
    and (root / 'dire_modal' / 'kaggle_celebahq_data.py').is_file()
]
if len(code_roots) != 1:
    listing = {str(mount): [child.name for child in mount.iterdir()][:12] for mount in mount_roots}
    raise FileNotFoundError(f'Expected one attached code package containing dire_modal; found {code_roots}. Mount listing: {listing}')
CODE_DATASET_ROOT = code_roots[0]
if not DIRE_ROOT.is_dir():
    raise FileNotFoundError(f'Missing DIRE input root: {DIRE_ROOT}')
if not SPLIT_ARTIFACT.is_file():
    raise FileNotFoundError(f'Missing Stage 3 v2 split artifact: {SPLIT_ARTIFACT}')

sys.path.insert(0, str(CODE_DATASET_ROOT))
from dire_modal.kaggle_celebahq_data import build_celebahq_datasets

print('Code package root added to sys.path:', CODE_DATASET_ROOT)
print('DIRE root:', DIRE_ROOT)
print('Split artifact:', SPLIT_ARTIFACT)

In [ ]:
from collections import Counter
import inspect
import json

def progress(message: str) -> None:
    print('[binding]', message, flush=True)

builder_kwargs = {}
if 'progress_callback' in inspect.signature(build_celebahq_datasets).parameters:
    builder_kwargs['progress_callback'] = progress
else:
    print('Attached code package lacks progress callback; scan still runs without intermediate progress.')

bundle = build_celebahq_datasets(DIRE_ROOT, SPLIT_ARTIFACT, **builder_kwargs)

train_counts = Counter(record.label for record in bundle.train.records)
validation_counts = Counter(record.label for record in bundle.validation.records)
test_counts = {name: len(dataset) for name, dataset in bundle.test_by_generator.items()}

expected_train = {0: 25_200, 1: 36_000}
expected_validation = {0: 2_800, 1: 4_000}
# Inventory counts previously reported for this mounted CelebA-HQ test tree.
expected_test = {
    'real': 1_000, 'dalle2': 500, 'if': 1_000,
    'midjourney': 100, 'stargan': 1_000, 'sdv2': 1_000,
}

assert dict(train_counts) == expected_train, (dict(train_counts), expected_train)
assert dict(validation_counts) == expected_validation, (dict(validation_counts), expected_validation)
assert test_counts == expected_test, (test_counts, expected_test)
assert bundle.validation_id_list_sha256 == (
    'c635c38a5888ef4e0196423dd8f342f412dc15ea633c17d835330c740d7dc9fd'
)

report = {
    'status': 'pass',
    'protocol': 'celebahq-content-disjoint-exact-count-hash-rank-v2',
    'source_fingerprint': bundle.source_fingerprint,
    'validation_id_list_sha256': bundle.validation_id_list_sha256,
    'train_count_by_label': {str(k): v for k, v in sorted(train_counts.items())},
    'validation_count_by_label': {str(k): v for k, v in sorted(validation_counts.items())},
    'test_count_by_generator': test_counts,
    'image_contents_read': False,
    'input_write_attempted': False,
    'gpu_workload_started': False,
    'training_started': False,
}
report_path = Path('/kaggle/working/celebahq-data-binding-report.json')
report_path.write_text(json.dumps(report, indent=2) + '\n', encoding='utf-8')
print(json.dumps(report, indent=2))
print('Saved report:', report_path)